# JAX Runner: Efficient Local Attention + Span-Hypergraph + Compressed Memory LM

Trains the pure-JAX implementation in `jax_model/` on a Hugging Face **streaming**
dataset (same pipeline as the original torch notebook: documents are tokenized on the
fly and packed into contiguous `block_size + 1` token chunks — nothing is downloaded
up front).

Notes for this JAX runner:

- Parameters are initialized by building the torch model once and converting with
  `jax_model.from_torch_model` (the repo's tested parity path); torch is not used after that.
- The optimizer is a self-contained AdamW + cosine schedule + global-norm clipping in
  pure JAX (no extra dependencies on top of the repo's pinned `jax==0.4.34` / jax-metal stack).
- `attention_backend="chunked"` is O(T·window) and is the only attention backend whose
  backward pass compiles on Apple Metal. The JAX forward has no dropout (deterministic).

## 1. Imports and configuration

In [ ]:
import math
import pickle
import time
from dataclasses import asdict

import jax
import jax.numpy as jnp
import numpy as np

import jax_model

print("jax devices:", jax.devices())
print("jax backend:", jax.default_backend())

seed = 1337

# -----------------------
# User-editable settings
# -----------------------

# Dataset: FineWeb-Edu sample-10BT, streamed. Swap to e.g. "roneneldan/TinyStories"
# (dataset_config=None) for a smaller/debug dataset.
dataset_name = "HuggingFaceFW/fineweb-edu"
dataset_config = "sample-10BT"
dataset_split = "train"
text_field = "text"
tokenizer_name = "gpt2"
shuffle_buffer = 50_000
val_docs = 2_000  # held out by taking the first docs from the stream

# Training shape. Scaled for Apple Metal (16 GB); raise on bigger hardware.
batch_size = 4
block_size = 1024

# Model size (same defaults as the torch notebook).
n_embd = 384
n_head = 6
n_local_attn_layers = 1
n_span_layers = 6
n_compressed_memory_layers = 1
span_widths = (2, 4, 8, 16, 32, 64)
local_window = 256
compression_block = 64

# Optimization.
max_iters = 10_000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-4
min_lr_ratio = 0.1
warmup_iters = 200
weight_decay = 0.1
grad_clip = 1.0
save_best_checkpoint = True
checkpoint_path = "best_jax_span_hypergraph_lm.pkl"

# JAX backends (see README): chunked attention trains on Metal and scales linearly.
attention_backend = "chunked"
span_backend = "materialized"

# Generation.
generate_tokens = 200
temperature = 0.8
top_k = 50
prompt = "The meaning of intelligence is"

np_rng = np.random.default_rng(seed)

## 2. Streaming packed-token dataset

Adapted from the original notebook's `StreamingPackedTokenDataset`, minus the torch
`DataLoader`: a plain generator that yields `(x, y)` NumPy int32 batches of shape
`(batch_size, block_size)`. The first `val_docs` documents are held out for validation;
the training stream skips them and shuffles with a buffer.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer


class StreamingPackedTokens:
    def __init__(self, shuffle, seed, skip_docs=0, take_docs=None):
        self.shuffle = shuffle
        self.seed = seed
        self.skip_docs = skip_docs
        self.take_docs = take_docs
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
        if self.tokenizer.eos_token_id is None:
            self.tokenizer.add_special_tokens({"eos_token": "<|endoftext|>"})
        self.eos_id = self.tokenizer.eos_token_id

    def _make_stream(self):
        if dataset_config is None:
            ds = load_dataset(dataset_name, split=dataset_split, streaming=True)
        else:
            ds = load_dataset(dataset_name, dataset_config, split=dataset_split, streaming=True)
        # Validation stream should be deterministic; training stream should be shuffled.
        if self.shuffle:
            ds = ds.shuffle(buffer_size=shuffle_buffer, seed=self.seed)
        if self.skip_docs:
            ds = ds.skip(self.skip_docs)
        if self.take_docs is not None:
            ds = ds.take(self.take_docs)
        return ds

    def _chunks(self):
        # Infinite stream for training; one finite pass when take_docs is set.
        while True:
            token_buffer = []
            yielded_any = False
            for row in self._make_stream():
                text = row.get(text_field, None)
                if not isinstance(text, str) or len(text) == 0:
                    continue
                ids = self.tokenizer.encode(text, add_special_tokens=False)
                ids.append(self.eos_id)
                token_buffer.extend(ids)
                while len(token_buffer) >= block_size + 1:
                    chunk = np.asarray(token_buffer[: block_size + 1], dtype=np.int32)
                    token_buffer = token_buffer[block_size + 1 :]
                    yielded_any = True
                    yield chunk[:-1], chunk[1:]
            if self.take_docs is not None:
                break
            if not yielded_any:
                raise RuntimeError("Dataset iterator yielded no examples. Check dataset config/text field.")

    def batches(self):
        xs, ys = [], []
        for x, y in self._chunks():
            xs.append(x)
            ys.append(y)
            if len(xs) == batch_size:
                yield np.stack(xs), np.stack(ys)
                xs, ys = [], []


train_ds = StreamingPackedTokens(shuffle=True, seed=seed, skip_docs=val_docs)
val_ds = StreamingPackedTokens(shuffle=False, seed=seed, take_docs=val_docs)
tokenizer = train_ds.tokenizer
vocab_size = len(tokenizer)
print("vocab_size:", vocab_size)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)


def cycle_batches(ds):
    while True:
        yield from ds.batches()


train_iter = cycle_batches(train_ds)
val_iter = cycle_batches(val_ds)

## 3. Model initialization

Build the torch model once with the repo's tested initialization, convert to a JAX
parameter pytree, then drop the torch model. `dropout=0.0` because the JAX forward
pass is deterministic.

In [ ]:
import torch

from model import EfficientHGConfig, EfficientHypergraphLM

torch.manual_seed(seed)
torch_cfg = EfficientHGConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_local_attn_layers=n_local_attn_layers,
    n_span_layers=n_span_layers,
    n_compressed_memory_layers=n_compressed_memory_layers,
    span_widths=span_widths,
    local_window=local_window,
    compression_block=compression_block,
    dropout=0.0,
)
params, cfg = jax_model.from_torch_model(EfficientHypergraphLM(torch_cfg))
print(f"parameters: {jax_model.count_parameters(params) / 1e6:.2f}M")
print("config:", asdict(cfg))

## 4. Optimizer and train/eval steps

Self-contained AdamW (decoupled weight decay on all parameters, like the torch
notebook's `torch.optim.AdamW(model.parameters(), ...)`), cosine LR schedule with
warmup, and global-norm gradient clipping. The whole update is one jitted step;
`lr` is a traced argument so the schedule does not trigger recompiles.

In [ ]:
def get_lr(step):
    if step < warmup_iters:
        return learning_rate * step / max(1, warmup_iters)
    if step > max_iters:
        return learning_rate * min_lr_ratio
    decay_ratio = (step - warmup_iters) / max(1, max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * min_lr_ratio + coeff * (learning_rate - learning_rate * min_lr_ratio)


def adamw_init(params):
    return {
        "m": jax.tree.map(jnp.zeros_like, params),
        "v": jax.tree.map(jnp.zeros_like, params),
        "step": jnp.zeros((), dtype=jnp.int32),
    }


def adamw_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    grad_norm = jnp.sqrt(sum(jnp.sum(jnp.square(g)) for g in jax.tree.leaves(grads)))
    clip_scale = jnp.minimum(1.0, grad_clip / (grad_norm + 1e-12))
    grads = jax.tree.map(lambda g: g * clip_scale, grads)

    step = state["step"] + 1
    m = jax.tree.map(lambda m, g: beta1 * m + (1 - beta1) * g, state["m"], grads)
    v = jax.tree.map(lambda v, g: beta2 * v + (1 - beta2) * jnp.square(g), state["v"], grads)
    bc1 = 1 - beta1 ** step.astype(jnp.float32)
    bc2 = 1 - beta2 ** step.astype(jnp.float32)
    params = jax.tree.map(
        lambda p, m, v: p - lr * ((m / bc1) / (jnp.sqrt(v / bc2) + eps) + weight_decay * p),
        params, m, v,
    )
    return params, {"m": m, "v": v, "step": step}, grad_norm


def loss_fn(p, x, y):
    return jax_model.loss(p, x, y, cfg, attention_backend=attention_backend, span_backend=span_backend)


@jax.jit
def train_step(params, opt_state, x, y, lr):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    params, opt_state, grad_norm = adamw_update(params, grads, opt_state, lr)
    return params, opt_state, loss, grad_norm


eval_step = jax.jit(loss_fn)


def estimate_loss():
    losses = []
    for _ in range(eval_iters):
        xb, yb = next(val_iter)
        losses.append(eval_step(params, jnp.asarray(xb), jnp.asarray(yb)))
    return float(jnp.mean(jnp.stack(losses)))


def print_gates(params):
    for i, block in enumerate(params["blocks"]):
        if "gate" in block:
            raw = float(block["gate"])
            print(f"blocks.{i:2d}.gate  raw={raw:+.4f} sigmoid={1 / (1 + math.exp(-raw)):.4f}")


opt_state = adamw_init(params)

## 5. Forward/backward smoke test

The first call compiles (slow); the second shows the steady-state step time.

In [ ]:
xb, yb = next(train_iter)
xb, yb = jnp.asarray(xb), jnp.asarray(yb)
print("x:", xb.shape, "y:", yb.shape)

t0 = time.time()
params, opt_state, loss, grad_norm = train_step(params, opt_state, xb, yb, get_lr(1))
loss.block_until_ready()
print(f"first step (incl. compile): {time.time() - t0:.1f}s  loss={float(loss):.4f}")

t0 = time.time()
params, opt_state, loss, grad_norm = train_step(params, opt_state, xb, yb, get_lr(2))
loss.block_until_ready()
dt = time.time() - t0
print(f"steady-state step: {dt * 1000:.0f}ms  ({xb.size / dt:,.0f} tok/s)  "
      f"loss={float(loss):.4f}  grad_norm={float(grad_norm):.3f}")

## 6. Training loop

In [ ]:
best_val = float("inf")
loss_ema = None
tokens_since_eval = 0
total_tokens = 0
t0 = time.time()

for step in range(max_iters + 1):
    if step % eval_interval == 0 or step == max_iters:
        elapsed = time.time() - t0
        toks_per_sec = 0.0 if step == 0 else tokens_since_eval / max(elapsed, 1e-9)
        val_loss = estimate_loss()
        train_loss_str = "nan" if loss_ema is None else f"{loss_ema:.4f}"
        print(
            f"step {step:6d} | train_ema {train_loss_str} | val {val_loss:.4f} | "
            f"lr {get_lr(step):.2e} | tok/s {toks_per_sec:,.0f} | "
            f"tokens {total_tokens:,} | elapsed {elapsed:.1f}s"
        )
        print_gates(params)
        t0 = time.time()
        tokens_since_eval = 0

        if save_best_checkpoint and val_loss < best_val:
            best_val = val_loss
            with open(checkpoint_path, "wb") as f:
                pickle.dump(
                    {
                        "params": jax.tree.map(np.asarray, params),
                        "config": asdict(cfg),
                        "step": step,
                        "val_loss": val_loss,
                        "train_loss_ema": loss_ema,
                        "total_tokens": total_tokens,
                        "tokenizer_name": tokenizer_name,
                    },
                    f,
                )
            print(f"saved best checkpoint to {checkpoint_path} with val={best_val:.4f}")

    if step == max_iters:
        break

    xb, yb = next(train_iter)
    params, opt_state, loss, grad_norm = train_step(
        params, opt_state, jnp.asarray(xb), jnp.asarray(yb), get_lr(step)
    )

    loss_value = float(loss)
    if not math.isfinite(loss_value):
        print(f"non-finite loss at step {step}: {loss_value} (grad_norm={float(grad_norm):.3f}) — stopping")
        break
    loss_ema = loss_value if loss_ema is None else 0.99 * loss_ema + 0.01 * loss_value
    tokens_since_eval += xb.size
    total_tokens += xb.size

## 7. Generate text

All blocks are causal, so generation runs the model over a fixed-size buffer (one jit
compile) and reads the logits at the current position; the buffer slides once full.

In [ ]:
# Optionally load the best checkpoint before generation:
# with open(checkpoint_path, "rb") as f:
#     ckpt = pickle.load(f)
# params = jax.tree.map(jnp.asarray, ckpt["params"])

gen_window = min(block_size, 512)


@jax.jit
def gen_logits(p, idx):
    return jax_model.forward(p, idx, cfg, attention_backend=attention_backend, span_backend=span_backend)


def generate(params, prompt, max_new_tokens, temperature=0.8, top_k=50, seed=0):
    rng = np.random.default_rng(seed)
    ids = tokenizer.encode(prompt)
    out = list(ids)
    ids = ids[-gen_window:]
    buf = np.full((1, gen_window), tokenizer.eos_token_id, dtype=np.int32)
    buf[0, : len(ids)] = ids
    cur = len(ids)

    for _ in range(max_new_tokens):
        logits = np.asarray(gen_logits(params, jnp.asarray(buf)))[0, cur - 1] / temperature
        if top_k is not None:
            cutoff = np.partition(logits, -top_k)[-top_k]
            logits = np.where(logits < cutoff, -np.inf, logits)
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        nxt = int(rng.choice(len(probs), p=probs))
        out.append(nxt)
        if cur == gen_window:
            buf[0, :-1] = buf[0, 1:]
            cur -= 1
        buf[0, cur] = nxt
        cur += 1
    return tokenizer.decode(out)


print(generate(params, prompt, generate_tokens, temperature=temperature, top_k=top_k, seed=seed))